# Exploracao da base de YouTube Shorts

Notebook de leitura local: abre as coletas de **moda** e **automotivo**, sem API e sem chave. Execute todas as celulas em ordem.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
RUNS = {'moda': '20260924T163050Z', 'automotivo': '20260924T163415Z'}
MIN_SHORTS = 30
COLORS = {'moda': '#7b4f72', 'automotivo': '#28666e', 'brand': '#4c78a8', 'creator': '#f58518'}

eligible_parts, profile_parts = [], []
for sector, run_id in RUNS.items():
    folder = ROOT / 'data' / 'silver' / 'data_discovery' / run_id
    eligible_parts.append(pd.read_csv(folder / 'eligible_shorts.csv'))
    profile_parts.append(pd.read_csv(folder / 'channel_profile.csv'))

eligible = pd.concat(eligible_parts, ignore_index=True)
profiles = pd.concat(profile_parts, ignore_index=True)
eligible['published_at'] = pd.to_datetime(eligible['published_at'], utc=True)
eligible['like_count'] = pd.to_numeric(eligible['like_count'], errors='coerce')
approved = profiles.query('eligible_shorts >= @MIN_SHORTS').copy()
history = eligible.merge(approved[['channel_id']], on='channel_id', how='inner')

print(f'Shorts elegiveis: {len(eligible):,}')
print(f'Canais aprovados: {len(approved)}')
print(f'Shorts dos canais aprovados: {len(history):,}')

## 1. Quantidade por area

In [ ]:
by_sector = eligible.groupby('sector')['video_id'].nunique().reindex(['moda', 'automotivo'])
fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(by_sector.index.str.title(), by_sector.values, color=[COLORS[x] for x in by_sector.index])
ax.bar_label(bars, padding=4)
ax.set_title('Shorts elegiveis por setor')
ax.set_ylabel('Quantidade de Shorts')
ax.spines[['top', 'right']].set_visible(False)
plt.show()
display(by_sector.rename('shorts_elegiveis').reset_index())

## 2. Quantidade de Shorts por canal

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
for ax, sector in zip(axes, ['moda', 'automotivo']):
    data = profiles.loc[profiles['sector'].eq(sector)].sort_values('eligible_shorts')
    labels = data['channel_name'] + ' (' + data['channel_type'] + ')'
    bars = ax.barh(labels, data['eligible_shorts'], color=data['channel_type'].map(COLORS))
    ax.bar_label(bars, padding=3, fontsize=8)
    ax.axvline(30, color='#6b7280', linestyle='--', label='minimo: 30')
    ax.axvline(50, color='#111827', linestyle=':', label='preferencia: 50')
    ax.set_title(sector.title())
    ax.set_xlabel('Shorts elegiveis')
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(loc='lower right')
fig.suptitle('Volume de Shorts por canal', y=1.02, fontsize=15)
plt.tight_layout()
plt.show()

## 3. Historico de curtidas e validade da variacao

Cada ponto e um Short. A escala de curtidas e logaritmica para comparar canais de tamanhos diferentes. A linha tracejada e a mediana do canal.

In [ ]:
def plot_history(sector, columns):
    channels = approved.loc[approved['sector'].eq(sector)].sort_values('eligible_shorts', ascending=False)
    rows = -(-len(channels) // columns)
    fig, axes = plt.subplots(rows, columns, figsize=(5.2 * columns, 3.5 * rows), squeeze=False)
    for ax, (_, channel) in zip(axes.ravel(), channels.iterrows()):
        data = history.loc[history['channel_id'].eq(channel['channel_id'])].sort_values('published_at')
        median = data['like_count'].median()
        ax.scatter(data['published_at'], data['like_count'] + 1, s=18, alpha=.7, color=COLORS[channel['channel_type']])
        ax.axhline(median + 1, color='#111827', linestyle='--', linewidth=1)
        ax.set_yscale('log')
        ax.set_title(f"{channel['channel_name']} | n={len(data)} | mediana={median:.0f}", fontsize=10)
        ax.set_xlabel('Publicacao')
        ax.set_ylabel('Curtidas (+1, log)')
        ax.spines[['top', 'right']].set_visible(False)
    for ax in axes.ravel()[len(channels):]: ax.set_visible(False)
    fig.suptitle(f'{sector.title()}: historico de curtidas', y=1.01, fontsize=15)
    plt.tight_layout()
    plt.show()

plot_history('moda', 3)
plot_history('automotivo', 4)

In [ ]:
validity = (
    history.groupby(['sector', 'channel_name', 'channel_type'])
    .agg(shorts=('video_id', 'nunique'), inicio=('published_at', 'min'), fim=('published_at', 'max'), curtidas_unicas=('like_count', 'nunique'), mediana=('like_count', 'median'), iqr=('like_count', lambda x: x.quantile(.75) - x.quantile(.25)), minimo=('like_count', 'min'), maximo=('like_count', 'max'))
    .reset_index()
)
validity['dias_cobertos'] = (validity['fim'] - validity['inicio']).dt.days
validity['variacao_observavel'] = validity['curtidas_unicas'].ge(10) & validity['iqr'].gt(0) & validity['dias_cobertos'].ge(90)
display(validity.sort_values(['sector', 'shorts'], ascending=[True, False]))
print(f"Canais com variacao observavel: {validity['variacao_observavel'].sum()} de {len(validity)}")